# Retrieval Method Comparison

This notebook compares the retrieval methods that now coexist in the demo app:

- `maid_bruteforce_sinter`
- `maid_tightened_sinter`
- `precomputed_segment`
- `hybrid_precompute_plus_realtime`
- `hybrid_bitmap_gating`
- `hybrid_bitmap_taxonomy`
- `full_realtime` as the correctness baseline

The focus here is not HTTP latency. It is the Redis work and the in-app decision path needed to get from an incoming identity token to validated candidates.

In [ ]:
from pathlib import Path
import json
import pandas as pd

reports_dir = Path('../reports/generated')
benchmark = json.loads((reports_dir / 'hybrid_benchmark.json').read_text(encoding='utf-8'))
report_text = Path('../reports/benchmark_report.md').read_text(encoding='utf-8')
list(benchmark.keys())

## 1. Definitions

The report now uses two related latency definitions:

- `decision_path_latency` = `identity_resolution_ms + profile_fetch_ms + candidate_generation_ms + campaign_fetch_ms + filtering_ms + rerank_ms`
- `validated_candidate_latency` = `candidate_generation_ms + campaign_fetch_ms + filtering_ms`

Both exclude HTTP/framework overhead. The first is the closer measure of end-to-end in-app decision work. The second isolates candidate production and validation after identity resolution is already done.

In [ ]:
overview = pd.DataFrame(benchmark['loadtests']).T[[
    'avg_sinter_ops',
    'avg_mode_redis_round_trips',
    'decision_path_p50_latency_ms',
    'decision_path_p99_latency_ms',
    'validated_candidate_p50_latency_ms',
    'validated_candidate_p99_latency_ms',
    'candidate_generation_avg_latency_ms',
    'candidate_generation_p99_latency_ms',
]].loc[[
    'maid_bruteforce_sinter',
    'maid_tightened_sinter',
    'precomputed_segment',
    'hybrid_precompute_plus_realtime',
    'hybrid_bitmap_gating',
    'full_realtime',
]]
overview

## 2. Method Definitions

- `maid_bruteforce_sinter`: the original MAID planner with 26 sequential `SINTER` probes.
- `maid_tightened_sinter`: a reduced MAID planner with only three pipelined `SINTER` probes.
- `precomputed_segment`: direct per-MAID precomputed candidate lists with `maid_hot` reranking.
- `hybrid_precompute_plus_realtime`: the same direct per-MAID list plus live mutable gating before reranking.
- `hybrid_bitmap_gating`: direct per-MAID list plus a server-side bitmap gate for active, pacing, and budget before campaign fetch.
- `hybrid_bitmap_taxonomy`: the bitmap-gated path plus app-side evaluation of each campaign's `taxonomy_filter` against the MAID's float interest scores.
- `full_realtime`: materialize the entire campaign universe and filter it live.

In [ ]:
method_rows = pd.DataFrame(
    [
        {'mode': 'maid_bruteforce_sinter', 'shape': 'identity -> maid -> 26 SINTERs -> campaign/state + fcap hash -> filter -> rerank'},
        {'mode': 'maid_tightened_sinter', 'shape': 'identity -> maid -> 3 SINTERs -> campaign/state + fcap hash -> filter -> rerank'},
        {'mode': 'precomputed_segment', 'shape': 'identity -> maid_hot -> aud:{maid} -> campaign/state + fcap hash -> minimal live gating -> rerank'},
        {'mode': 'hybrid_precompute_plus_realtime', 'shape': 'identity -> maid_hot -> aud:{maid} -> campaign/state + fcap hash -> live mutable gating -> rerank'},
        {'mode': 'hybrid_bitmap_gating', 'shape': 'identity -> maid_hot -> aud:{maid} -> bm:servable gate -> campaign + fcap hash -> frequency check -> rerank'},
        {'mode': 'full_realtime', 'shape': 'identity -> maid -> full campaign universe -> filter everything live -> rerank'},
    ]
)
method_rows

## 3. Reading The Current Results

The main progression is:

1. reduce the number of `SINTER` probes,
2. reduce Redis round trips,
3. remove hot-path set algebra entirely by moving static targeting into batch,
4. then decide which exact checks still need to run live.

That is why the direct precompute modes now beat the MAID `SINTER` modes on both p50 and p99 decision-path latency. The bitmap-gated variants are the current experiments for reducing live mutable-state fanout further, and `hybrid_bitmap_taxonomy` is the low-fanout path that still preserves the per-ad float-threshold logic.

In [ ]:
comparison = overview.copy()
comparison['decision_path_gain_vs_bruteforce_p99_ms'] = comparison.loc['maid_bruteforce_sinter', 'decision_path_p99_latency_ms'] - comparison['decision_path_p99_latency_ms']
comparison[['decision_path_p99_latency_ms', 'decision_path_gain_vs_bruteforce_p99_ms']]

## 4. Takeaway

The notebook and the benchmark report now tell the same story:

- brute-force MAID `SINTER` retrieval is the cautionary starting point,
- tightened `SINTER` retrieval is a meaningful improvement but still has ugly tail behavior,
- direct per-MAID precompute removes hot-path set work and produces the best current latency profile in this demo.